In [ ]:
# check working directory as it helps to know where the notebook is running and how relative paths will be resolved
import os, sys
print(os.getcwd())
print(sys.path[:5])

/content
['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload']


In [13]:
from pathlib import Path
from datasets import load_dataset

output_path = Path("data/contemporary_corpus.txt")
output_path.parent.mkdir(parents=True, exist_ok=True)

if output_path.exists() and output_path.stat().st_size > 0:
    print(f"Using existing corpus file: {output_path.resolve()}")
else:
    print("Building contemporary_corpus.txt from AG News...")

    dataset = load_dataset("ag_news", split="train")
    first_column_row = dataset.column_names[0]
    print(dataset.column_names)
    print(dataset[0])

    with output_path.open("w", encoding="utf-8") as f:
        for row in dataset:
            text = row[first_column_row].strip()
            if text:
                f.write(text + "\n")

    # Creates at /content/data/contemporary_corpus.txt
    # which is a relative path. So the actual file location depends on the notebook kernel’s current working directory.
    print(f"Created corpus file: {output_path.resolve()}")

Using existing corpus file: /content/data/contemporary_corpus.txt


In [ ]:
# from config import CorpusConfig
# from pipeline import train_word2vec_pipeline
# from preprocessing import generate_training_data

from nlp.word_embeddings_pipeline_001.config import CorpusConfig
from nlp.word_embeddings_pipeline_001.pipeline import train_word2vec_pipeline
from nlp.word_embeddings_pipeline_001.preprocessing import generate_training_data
contemporary_config = CorpusConfig(
    corpus_name="contemporary_news",
    source_type="local_file",
    source="data/contemporary_corpus.txt",
    vocab_size=8000,
    # sequence_length=12,
    # window_size=2,
    # num_negative_samples=4,
    embedding_dim=128,
    batch_size=1024,
    # epochs=20,
    epochs=10,
    # epochs=1,  # for quick testing, but should be increased for better results
    lowercase=True,
    strip_punctuation=False,
    keep_apostrophes=True,
    # controls preprocessing
    sequence_length=5,
    window_size=1,
    num_negative_samples=2,
    min_line_length=20,
    max_lines=23000,
)

shakespeare_config = CorpusConfig(
    corpus_name="shakespeare",
    source_type="url",
    source="https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt",
    vocab_size=4096,
    # sequence_length=10,
    # window_size=2,
    # num_negative_samples=4,
    embedding_dim=128,
    batch_size=1024,
    buffer_size=10000,
    # epochs=20,
    epochs=10,
    # epochs=1,  # for quick testing, but should be increased for better results
    seed=42,
    lowercase=True,
    strip_punctuation=True,
    keep_apostrophes=False,
    # min_line_length=1,
    # controls preprocessing
    sequence_length=5,
    window_size=2,
    num_negative_samples=2,
    min_line_length=20,
)


def print_training_results(config, results):
  word2vec = results["model"]
  w2v_history = results["history"]
  weights = results["weights"]
  vocab = results["vocab"]
  vectorize_layer = results["vectorize_layer"]
  dataset = results["dataset"]
  sequences = results["sequences"]
  targets = results["targets"]
  contexts = results["contexts"]
  labels = results["labels"]

  print(f"Corpus: {config.corpus_name}")
  print(f"Number of sequences: {len(sequences)}")
  print(f"Number of training examples: {len(targets)}")
  print(f"targets.shape: {targets.shape}")
  print(f"contexts.shape: {contexts.shape}")
  print(f"labels.shape: {labels.shape}")
  print(f"Embedding shape: {weights.shape}")
  print(f"First 10 vocab terms: {vocab[:10]}")

ModuleNotFoundError: No module named 'nlp'

In [ ]:
shakespeare_results = train_word2vec_pipeline(
    config=shakespeare_config,
    generate_training_data=generate_training_data,
)

contemporary_results = train_word2vec_pipeline(
    config=contemporary_config,
    generate_training_data=generate_training_data,
)

results_by_name = {
    "Shakespeare": shakespeare_results,
    "Contemporary": contemporary_results,
}

print_training_results(shakespeare_config, shakespeare_results)
print_training_results(contemporary_config, contemporary_results)

In [ ]:
import matplotlib.pyplot as plt

def plot_training_comparison(results_by_name):
    fig = plt.figure(figsize=(12, 5))
    ax_loss, ax_acc = fig.subplots(1, 2)

    for name, results in results_by_name.items():
        history = results["history"].history
        loss = history.get("loss", [])
        accuracy = history.get("accuracy", [])

        epochs = range(1, len(loss) + 1)

        ax_loss.plot(epochs, loss, marker="o", label=name)
        ax_acc.plot(epochs, accuracy, marker="o", label=name)

    ax_loss.set_title("Training Loss")
    ax_loss.set_xlabel("Epoch")
    ax_loss.set_ylabel("Loss")
    ax_loss.legend()

    ax_acc.set_title("Training Accuracy")
    ax_acc.set_xlabel("Epoch")
    ax_acc.set_ylabel("Accuracy")
    ax_acc.legend()

    fig.suptitle("Word2Vec Training by Corpus")
    fig.tight_layout()
    plt.show()

plot_training_comparison({
    "Shakespeare": shakespeare_results,
    "Contemporary": contemporary_results,
})

In [ ]:
def plot_final_metrics(results_by_name):
    names = []
    final_loss = []
    final_accuracy = []

    for name, results in results_by_name.items():
        history = results["history"].history
        names.append(name)
        final_loss.append(history["loss"][-1])
        final_accuracy.append(history["accuracy"][-1])

    fig, ax = plt.subplots(1, 2, figsize=(12, 5))

    ax[0].bar(names, final_loss)
    ax[0].set_title("Final Loss")
    ax[0].set_ylabel("Loss")

    ax[1].bar(names, final_accuracy)
    ax[1].set_title("Final Accuracy")
    ax[1].set_ylabel("Accuracy")

    fig.suptitle("Final Metrics by Corpus")
    plt.tight_layout()
    plt.show()

plot_final_metrics({
    "Shakespeare": shakespeare_results,
    "Contemporary": contemporary_results,
})

In [ ]:
import matplotlib.pyplot as plt

def plot_training_comparison(results_by_name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for name, results in results_by_name.items():
        history = results["history"].history
        axes[0].plot(history["loss"], marker="o", label=name)
        axes[1].plot(history["accuracy"], marker="o", label=name)

    axes[0].set_title("Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()

    axes[1].set_title("Training Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].legend()

    fig.suptitle("Word2Vec Training by Corpus")
    plt.tight_layout()
    plt.show()

plot_training_comparison(results_by_name)

In [ ]:
def plot_final_metrics(results_by_name):
    names = []
    final_losses = []
    final_accuracies = []

    for name, results in results_by_name.items():
        history = results["history"].history
        names.append(name)
        final_losses.append(history["loss"][-1])
        final_accuracies.append(history["accuracy"][-1])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].bar(names, final_losses)
    axes[0].set_title("Final Loss")
    axes[0].set_ylabel("Loss")

    axes[1].bar(names, final_accuracies)
    axes[1].set_title("Final Accuracy")
    axes[1].set_ylabel("Accuracy")

    fig.suptitle("Final Training Metrics by Corpus")
    plt.tight_layout()
    plt.show()

plot_final_metrics(results_by_name)

In [ ]:

def get_nearest_neighbors(results, query_word, top_k=5):
    vocab = results["vocab"]
    weights = results["weights"]

    if query_word not in vocab:
        return []

    word_index = vocab.index(query_word)
    query_vector = weights[word_index]

    norms = np.linalg.norm(weights, axis=1, keepdims=True)
    normalized_weights = weights / np.clip(norms, 1e-12, None)

    query_norm = np.linalg.norm(query_vector)
    query_vector = query_vector / max(query_norm, 1e-12)

    scores = normalized_weights @ query_vector
    nearest_indices = np.argsort(scores)[::-1]

    neighbors = []
    for index in nearest_indices:
        candidate = vocab[index]
        if candidate == query_word:
            continue
        if candidate in ("", "[UNK]"):
            continue
        neighbors.append((candidate, float(scores[index])))
        if len(neighbors) == top_k:
            break

    return neighbors

In [ ]:
# use king, love, man, woman, day, or time 
query_word = "king"

for name, results in results_by_name.items():
    neighbors = get_nearest_neighbors(results, query_word=query_word, top_k=8)
    print(f"\n{name} neighbors for '{query_word}':")
    for word, score in neighbors:
        print(f"{word:15s} {score:.4f}")